# Example 5: batch fitting and throughput triage

This notebook shows the pattern for handling tens to hundreds of spectra: run a cheap dry-run/audit first, then only launch PHOENIX fits when the setup and quality flags look reasonable.

The default notebook does not fit anything.

## What this example teaches

- how to plan a small batch before spending PHOENIX time;
- how quickscan, checkpoint/resume, and quality-gated refinement fit together;
- how to summarize throughput without treating every numerical result as science-ready.

## Requirements

The default dry-run uses bundled spectra only. Real fitting requires PHOENIX and should be launched deliberately with the script or fit-enabled cells.

## Expected outputs

A dry-run plan, setup/readiness summaries for each target, and—when fitting is enabled—JSON/CSV checkpoint products, throughput counts, and a few representative fit-inspection plots.


Note: Batch throughput success only means the workflow ran and preserved provenance. It is not a validation of atmospheric accuracy, and it does not replace inspection of flagged spectra or representative residual plots.


## 1. Define a tiny bundled batch

We use three bundled Gaia benchmark spectra so the notebook is reproducible: two ordinary standards and one metal-poor stress case. In real work, these paths would come from your target list or manifest.


In [ ]:
import numpy as np
import Spyctres as sp

# Three small bundled benchmark spectra: two ordinary standards plus one
# metal-poor stress target that should be interpreted separately.
spectra = [
    sp.example_data_path("gaia_benchmark/HIP79672_HARPS_1_R42KNorm.txt.gz"),
    sp.example_data_path("gaia_benchmark/HIP37279_HARPS_1_R42KNorm.txt.gz"),
    sp.example_data_path("gaia_benchmark/HIP76976_HARPS_1_R42KNorm.txt.gz"),
]

# All three files use the Gaia Benchmark Stars v3 ASCII reader.
reader = "gbs_v3_ascii"

for path in spectra:
    print(path.name)


## 2. Dry-run the batch plan without PHOENIX

This cheap loop reads each spectrum, asks Spyctres for a first-pass setup, and records whether the target looks fit-ready. This is the step to run before spending time on a large PHOENIX batch.


In [ ]:
batch_plan = []
for path in spectra:
    # Read one target.
    spec = sp.read_spectrum(path, reader=reader)

    # Ask Spyctres for a lightweight quicklook setup.
    setup = sp.suggest_fit_setup(
        spec,
        mode="quicklook",
        intent="quicklook_classification",
    )

    # Audit the exact regions from the reviewed setup.
    audit = sp.audit_spectrum_for_fit(
        spec,
        regions=setup.regions,
        intent="quicklook_classification",
    )

    # Keep a compact row for the notebook display.
    batch_plan.append(
        {
            "name": path.name,
            "reader": reader,
            "ready": audit["ready_for_intent"],
            "n_fit_pixels": audit["n_fit_candidate"],
            "flags": audit["interpretation_flags"],
            "setup_mode": setup.mode,
            "window": setup.summary()["recommended_window_label"],
        }
    )

batch_plan


## 3. Inspect representative targets before launching a batch fit

For a real batch, inspect a few clean and a few problematic spectra first. This catches reader, continuum, masking, or resolution issues before you fit dozens of spectra.


In [ ]:
# Plot diagnostic windows for the first target as a representative check.
first_spec = sp.read_spectrum(spectra[0], reader=reader)
first_windows = sp.select_diagnostic_windows(first_spec, max_windows=4)

sp.plot_spectrum_line_windows(
    first_spec.wave,
    first_spec.flux,
    first_windows.selected,
    valid_mask=first_spec.valid_mask,
    title="Example 5: representative batch target windows",
    ncols=2,
    figsize_per_panel=(7.2, 3.4),
)


## 4. Optional quickscan fit loop

Leave this off during a first read-through. When enabled, it is still a quicklook pass. For a real batch, prefer the checkpointing script in the next section.


In [ ]:
# Set to True only after PHOENIX is configured and you want a quicklook fit loop.
RUN_QUICKSCAN = True

quick_results = []
quick_specs = []
quick_labels = []

if RUN_QUICKSCAN:
    for path in spectra:
        # Keep the spectrum object so the next cell can plot the same target.
        spec = sp.read_spectrum(path, reader=reader)
        setup = sp.suggest_fit_setup(
            spec,
            mode="quicklook",
            intent="quicklook_classification",
        )
        result = sp.fit_stellar_spectrum(
            spec,
            model="phoenix",
            setup=setup,
            reconstruct=True,  # needed for the representative plots below
        )
        quick_specs.append(spec)
        quick_results.append(result)
        quick_labels.append(path.name)
        print(path.name)
        print("------")
        print(result.summary_text(include_hash=False, max_flags=5))

    quick_comparison = sp.compare_fits(quick_results, labels=quick_labels)
    print(sp.format_fit_comparison_table(quick_comparison))
else:
    print("RUN_QUICKSCAN is False; the notebook stopped after the cheap dry-run plan.")


## 5. Inspect representative fitted windows

For a real batch, do not inspect only the table. Plot a small representative subset: usually one ordinary successful fit and one flagged or high-χ² target. This keeps the notebook readable while still teaching the habit needed for tens or hundreds of spectra.

The cell below uses the quickscan results already in memory. It does not refit anything; it only plots model/data/residual windows for a few selected targets.


In [ ]:
# How many quickscan fits should the notebook display?
# For a large batch, keep this small and inspect saved plot files instead.
MAX_REPRESENTATIVE_PLOTS = 2

if RUN_QUICKSCAN and quick_results:
    chi2_values = [
        float(getattr(result, "chi2_red", float("nan")))
        for result in quick_results
    ]

    # Start with the first target, then add the highest-χ² target if it differs.
    representative_indices = [0]
    finite_indices = [
        index for index, value in enumerate(chi2_values)
        if np.isfinite(value)
    ]
    if finite_indices:
        worst_index = max(finite_indices, key=lambda index: chi2_values[index])
        if worst_index not in representative_indices:
            representative_indices.append(worst_index)

    for index in representative_indices[:MAX_REPRESENTATIVE_PLOTS]:
        spec = quick_specs[index]
        result = quick_results[index]
        label = quick_labels[index]

        # Plot the same broad diagnostic windows used for batch triage.
        plot_windows = sp.select_diagnostic_windows(spec, max_windows=4).selected
        sp.plot_model_line_windows(
            result,
            windows=plot_windows,
            title=f"Example 5: representative quickscan fit — {label}",
            show_residuals=True,
            residual_kind="auto",
            ncols=2,
            figsize_per_panel=(7.2, 4.2),
            footer=(
                "Representative batch diagnostic. Use this to decide which "
                "targets need deeper review, not as an automatic pass/fail."
            ),
        )
else:
    print(
        "No quickscan fit plots to show yet. Set RUN_QUICKSCAN=True, run the "
        "fit cell above, then rerun this cell."
    )


## 6. Reproduce the operational workflow quickly

For real throughput tests, use a script. It checkpoints after every target and can resume, which is what you want for large batches.


In [ ]:
print("Dry run from a shell:")
print("python examples/example5_batch_fitting.py --dry-run")
print()
print("Quicklook checkpoint run with representative fit plots:")
print(
    "python examples/example5_batch_fitting.py --quicklook "
    "--output-json /tmp/spyctres_example5_batch_quick.json "
    "--summary-csv /tmp/spyctres_example5_batch_quick.csv "
    "--plot-dir /tmp/spyctres_example5_plots "
    "--max-plots 2 --resume"
)
print()
print("Summarize throughput:")
print("python scripts/throughput_summary.py /tmp/spyctres_example5_batch_quick.json --project 100")


## 7. What to try next

After you manually inspect representative plots, the next layer is benchmark validation: keep clean benchmark stars separate from stress spectra, report aggregate bias/scatter, and avoid tuning per star.


In [ ]:
sp.describe_public_function("suggest_fit_setup")
